## Read raw data into pandas

In [11]:
import pandas as pd

raw = pd.read_csv("../data/raw/raw_profiles_small2.csv")
raw.head()

,profile_id,country,experience
0,10,romania,Codeless Technology B.V. | RomaniaCodeless Tec...


## Define the experience data class
The experience data class will correspond to one row in the resulting csv.

In [12]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class Experience:
    """Represents a single work experience entry from a LinkedIn profile.
    
    Designed to map directly to a CSV row.
    """
    profile_id: int
    job_title: str
    country: Optional[str]
    start_date: Optional[str]  # e.g. "2020-01" or "Jan 2020"
    end_date: Optional[str]    # None means present/current role
    company: Optional[str] = None  # Optional field for company name
    position_in_career: Optional[int] = None  # Optional field for position in career timeline

## Format the Experiences
Format the existing experiences in the format that matches the experience data class.

In [13]:
import os
import json
import time
import re
from dotenv import load_dotenv
from google import genai
from google.genai.errors import ClientError

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"  # alternatives: "gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-lite"

PARSE_PROMPT = """You are a structured data extractor. Given raw LinkedIn experience text for one person,
extract each distinct job/position into a JSON array. The text is messy — it may contain duplicated
fragments, merged lines, location strings, employment types (Full-time, Part-time, etc.), and
date strings that appear twice.

For each job, extract:
- "job_title": the cleaned job title (e.g. "Software Engineer", "IT Manager")
- "company": the company/organisation name, or null if unclear
- "start_date": in "YYYY-MM" format if month is available, or "YYYY" if only year, or null
- "end_date": in "YYYY-MM" format, "YYYY", or null if marked "Present" or still current
- "is_current": true if this is a current/present position, false otherwise

Rules:
- If the job title is in any other language than English, translate it to English if possible. If translation is uncertain, keep the original.
- Ignore location strings, employment type labels (Full-time, Part-time, Hybrid, Remote, On-site),
  logo mentions, and duration strings (e.g. "6 yrs 3 mos").
- If the same position appears to be duplicated (same title, same company, same dates), output it only once.
- If a company umbrella header appears (e.g. "ING Hubs Romania Full-time · 6 yrs 9 mos"),
  treat it as context for the sub-positions below it, not as a separate job.
- Return ONLY the JSON array, no markdown fences, no explanation.

Raw experience text:
{experience_text}

JSON array:"""

REQUESTS_PER_MINUTE = 13  # stay safely under free-tier 15/min
MAX_RETRIES = 3


def parse_experience_with_llm(profile_id: int, country: str, experience_text: str) -> list[Experience]:
    """Send raw experience text to Gemini and parse the JSON response into Experience objects.
    
    position_in_career is assigned so that 1 = oldest job, N = most recent job.
    LinkedIn lists jobs most-recent-first, so the first item in the array gets the highest index.
    """
    if pd.isna(experience_text) or not experience_text.strip():
        return []

    prompt = PARSE_PROMPT.format(experience_text=experience_text)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            raw_text = response.text.strip()
            # Strip markdown fences if present
            raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
            raw_text = re.sub(r"\s*```$", "", raw_text)

            jobs = json.loads(raw_text)
            n = len(jobs)
            experiences = []
            for idx, job in enumerate(jobs):
                experiences.append(Experience(
                    profile_id=profile_id,
                    job_title=job.get("job_title", "Unknown"),
                    country=country,
                    start_date=job.get("start_date"),
                    end_date=job.get("end_date") if not job.get("is_current") else None,
                    company=job.get("company"),
                    position_in_career=n - idx,  # most recent (first in list) → highest index
                ))
            return experiences

        except (json.JSONDecodeError, KeyError) as e:
            print(f"  JSON parse error for profile {profile_id} (attempt {attempt+1}): {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(5)
        except ClientError as e:
            if "429" in str(e):
                match = re.search(r"retry in (\d+)", str(e))
                wait = int(match.group(1)) + 5 if match else 60
                print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                time.sleep(wait)
            else:
                print(f"  API error for profile {profile_id}: {e}")
                break
        except Exception as e:
            print(f"  Unexpected error for profile {profile_id}: {e}")
            break

    # Fallback: return a single unknown entry so the profile isn't lost
    return [Experience(
        profile_id=profile_id,
        job_title="PARSE_ERROR",
        country=country,
        start_date=None,
        end_date=None,
        company=None,
        position_in_career=None,
    )]


# --- Run the LLM parser across all profiles ---
all_experiences: list[Experience] = []
delay = 60 / REQUESTS_PER_MINUTE

for i, row in raw.iterrows():
    pid = row["profile_id"]
    print(f"Parsing profile {pid} ({i+1}/{len(raw)})...")
    experiences = parse_experience_with_llm(pid, row["country"], row["experience"])
    all_experiences.extend(experiences)
    print(f"  -> {len(experiences)} jobs extracted")
    time.sleep(delay)

print(f"\nDone. Total experience entries: {len(all_experiences)}")
all_experiences[:5]

Parsing profile 10 (1/1)...
  -> 5 jobs extracted

Done. Total experience entries: 5


[Experience(profile_id=10, job_title='Low code developer', country='romania', start_date='2023-09', end_date=None, company='Codeless Technology B.V. | Romania', position_in_career=5),
 Experience(profile_id=10, job_title='Low code Internship', country='romania', start_date='2023-07', end_date='2023-09', company='Codeless Technology B.V. | Romania', position_in_career=4),
 Experience(profile_id=10, job_title='Student', country='romania', start_date='2020', end_date='2023', company='FEAA - Facultatea de Economie și Administrarea Afacerilor din Iași', position_in_career=3),
 Experience(profile_id=10, job_title='Saleswoman', country='romania', start_date='2022-07', end_date='2022-09', company='SC BSB FASHION SRL', position_in_career=2),
 Experience(profile_id=10, job_title='Saleswoman', country='romania', start_date='2021-06', end_date='2021-09', company='SC BSB FASHION SRL', position_in_career=1)]

## Export to csv

In [14]:
from dataclasses import asdict

output_path = "../data/processed/parsed_experiences_small2.csv"

df_experiences = pd.DataFrame([asdict(e) for e in all_experiences])
df_experiences.to_csv(output_path, index=False)

print(f"Saved {len(df_experiences)} rows to {output_path}")
df_experiences.head()

Saved 5 rows to ../data/processed/parsed_experiences_small2.csv


,profile_id,job_title,country,start_date,end_date,company,position_in_career
0,10,Low code developer,romania,2023-09,NaN,Codeless Technology B.V. | Romania,5
1,10,Low code Internship,romania,2023-07,2023-09,Codeless Technology B.V. | Romania,4
2,10,Student,romania,2020,2023,FEAA - Facultatea de Economie și Administrarea...,3
3,10,Saleswoman,romania,2022-07,2022-09,SC BSB FASHION SRL,2
4,10,Saleswoman,romania,2021-06,2021-09,SC BSB FASHION SRL,1


## Categorize Job Titles

In [15]:
df_experiences = pd.read_csv("../data/processed/parsed_experiences_small2.csv")
print(f"Loaded {len(df_experiences)} rows from parsed_experiences_small2.csv")

CLASSIFY_PROMPT = """
You are an expert in software development role classification. Your task is to classify a given job role into exactly one of the following three categories.

Categories

1. Traditional Software Development Roles that primarily involve writing code using traditional programming languages and software engineering practices. This includes, but is not limited to: Software Engineer, Developer, Backend Developer, Frontend Developer, Full Stack Developer, Mobile Developer, DevOps Engineer, Data Scientist, Network Engineer, Embedded Systems Developer, Systems Programmer.

2. Low-Code/No-Code (LCNC) Development Roles where the person primarily works with an LCNC platform to build applications with minimal or no traditional coding. This includes roles using platforms such as Power Platform (Power Apps, Power Automate, Power BI), Mendix, OutSystems, Appian, AppSheet, Webflow, Bubble, Framer, Zapier, Make (Integromat), and Airtable. Often found in a citizen developer, business analyst, or digital automation context.

3. Other Roles that do not clearly fit into either category above. This includes management, sales, design, HR, and unrelated technical roles.

Instructions Classify the given job role into exactly one of the three categories. Base your classification solely on the job title or role provided. If a role is ambiguous, choose the most likely category based on common industry usage. Return only the category name, nothing else.

Output format Return only one of these exact strings: "Traditional Software Development", "Low-Code/No-Code Development", or "Other".

Job title: {job_title}"""

VALID_CATEGORIES = {
    "Traditional Software Development",
    "Low-Code/No-Code Development",
    "Other",
}


def classify_job(job_title: str) -> str:
    """Classify a single job title into one of the four categories using Gemini."""
    prompt = CLASSIFY_PROMPT.format(job_title=job_title)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL, contents=prompt)
            label = response.text.strip()
            return label if label in VALID_CATEGORIES else "Other"

        except ClientError as e:
            if "429" in str(e):
                match = re.search(r"retry in (\d+)", str(e))
                wait = int(match.group(1)) + 5 if match else 60
                print(f"  Rate limited. Waiting {wait}s (attempt {attempt+1})...")
                time.sleep(wait)
            else:
                print(f"  API error for '{job_title}': {e}")
                break
        except Exception as e:
            print(f"  Unexpected error for '{job_title}': {e}")
            break

    return "ERROR"


# --- Classify all parsed experiences ---
labels = []
total = len(df_experiences)

for i, row in df_experiences.iterrows():
    title = row["job_title"]

    if title == "PARSE_ERROR":
        labels.append("ERROR")
        continue

    print(f"Classifying [{i+1}/{total}] '{title}'...")
    label = classify_job(title)
    labels.append(label)
    time.sleep(delay)

df_classified = df_experiences.copy()
df_classified["job_label"] = labels

output_classified = "../data/processed/classified_jobs_gemini_small2.csv"
df_classified.to_csv(output_classified, index=False)

print(f"\nDone. Saved {len(df_classified)} classified entries to {output_classified}")
df_classified.head(10)


Loaded 5 rows from parsed_experiences_small2.csv
Classifying [1/5] 'Low code developer'...
Classifying [2/5] 'Low code Internship'...
Classifying [3/5] 'Student'...
Classifying [4/5] 'Saleswoman'...
Classifying [5/5] 'Saleswoman'...

Done. Saved 5 classified entries to ../data/processed/classified_jobs_gemini_small2.csv


,profile_id,job_title,country,start_date,end_date,company,position_in_career,job_label
0,10,Low code developer,romania,2023-09,NaN,Codeless Technology B.V. | Romania,5,Low-Code/No-Code Development
1,10,Low code Internship,romania,2023-07,2023-09,Codeless Technology B.V. | Romania,4,Low-Code/No-Code Development
2,10,Student,romania,2020,2023,FEAA - Facultatea de Economie și Administrarea...,3,Other
3,10,Saleswoman,romania,2022-07,2022-09,SC BSB FASHION SRL,2,Other
4,10,Saleswoman,romania,2021-06,2021-09,SC BSB FASHION SRL,1,Other


## Ran 32 classifications as seen above. See pricing on Google Cloud and extrapolate to see what can be expected as token costs.